# React — Conditional rendering & lists

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> This topic has three playground experiments: `03-conditional.jsx` (LESSON 18),
> `04-lists.jsx` (LESSON 19) and `05-keys.jsx` (LESSON 20).

## LESSON 18 — Conditional rendering

Three shapes. Which one you reach for is decided by the UI, not by taste.

### 1. Early return — the whole component is different, or absent

```jsx
function Notice({ text }) {
  if (!text) return null;

  return <p>Notice: {text}</p>;
}
```

This is LESSON 8's shape: a statement runs **above** the `return`. Returning `null` from a
component means "render nothing at all" — the same rule as LESSON 10, applied to a whole
component rather than to one value.

### 2. Ternary — one of two things, inside a tree you are already building

```jsx
<p>Status: {online ? "Online" : "Offline"}</p>
```

Braces take an expression, and a ternary is an expression. The surrounding `<p>` renders
either way; only its contents change.

### 3. `&&` — something or nothing, inside a tree you are already building

```jsx
<div>
  <h2>Orders</h2>
  {hasDiscount && <Badge>Sale</Badge>}
</div>
```

**The distinction worth keeping straight:** an early return replaces the *entire* returned
tree. A ternary or `&&` includes or swaps *part* of a tree that renders regardless. If the
component still has to draw its wrapper, its heading and its layout, an early return is the
wrong tool.

### The `0` trap

`&&` does not return `true` or `false`. MDN:

> the operator returns the value of the first falsy operand encountered when evaluating from
> left to right, or the value of the last operand if they are all truthy

So `count && <Badge/>` **is** `count` whenever `count` is falsy. React then renders that
value, and LESSON 10's table decides what you see:

| falsy left-hand value | `&&` gives back | React renders |
|---|---|---|
| `false` | `false` | nothing |
| `null` | `null` | nothing |
| `undefined` | `undefined` | nothing |
| `""` | `""` | nothing |
| **`0`** | **`0`** | **`0`, on the page** |
| **`NaN`** | **`NaN`** | **`NaN`, on the page** |

Two of the six leak. And `0` is the one you will actually hit, because counts and lengths
are everywhere:

```jsx
{cart.count && <Badge>In your cart</Badge>}
```

An empty cart puts a bare `0` on the screen. React's docs say it flatly: **"Don't put
numbers on the left side of `&&`."**

**The fix is to hand `&&` a real boolean:**

```jsx
{cart.count > 0 && <Badge>In your cart</Badge>}
```

`cart.count > 0` is `true` or `false`, and both render nothing when false. A ternary with
`null` — `{cart.count > 0 ? <Badge/> : null}` — is equally safe, and never leaks at all,
because you choose both outcomes yourself.

### Choosing

| the UI needs | use |
|---|---|
| this component to render nothing, or something else entirely | early return |
| one of two things, inside a tree that always renders | ternary |
| something or nothing, inside a tree that always renders | `&&`, with a **boolean** on the left |
| more than two outcomes | work it out above the `return` (LESSON 8), put the result in braces |

### Key Notes

- Early return swaps the **whole** tree; ternary and `&&` swap a **part** of one.
- `&&` gives back the left value when it is falsy — it does not convert it to a boolean.
- Of the six falsy values, `0` and `NaN` are the two React renders. That is the whole trap.
- Put a comparison on the left — `count > 0` — not a number.

### Example

**Runnable — plain JS.** The trap is JavaScript, not React: the `&&` half happens before
React ever sees a value. Run this and read the middle column before the right one.

In [ ]:
// What React does with a value, from LESSON 10.
function l18renders(value) {
  if (value === null || value === undefined) return "nothing";
  if (typeof value === "boolean") return "nothing";
  if (value === "") return "nothing";
  return `the text ${JSON.stringify(String(value))}`;
}

const l18falsy = [false, null, undefined, "", 0, NaN];

for (const left of l18falsy) {
  const result = left && "<Badge/>"; // the guard, as plain JavaScript
  console.log(
    String(left).padEnd(10),
    "&& gives back:", String(result).padEnd(10),
    "-> React renders", l18renders(result),
  );
}

### Exercise

**Part 1.** Write `l18guard(left)`, which returns what a reader would actually see on the
page for `{left && <Badge/>}`. Exactly one of these strings:

| case | returns |
|---|---|
| the badge shows | `"badge"` |
| nothing visible | `"nothing"` |
| a stray value leaks onto the page | `"LEAK: <the value>"` |

Run it over `[false, null, undefined, "", 0, NaN, 3, "hello"]` and log each result. Two of
them must report a leak — and you should be able to say which two before you run it.

**Part 2.** Here are three carts:

```js
const l18carts = [{ count: 0 }, { count: 3 }, { count: NaN }];
```

For each one, log the value that `count > 0` produces, and confirm with `l18guard` that the
boolean guard leaks nothing for any of the three.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Part 1 — in the playground.** Point `playground/src/App.jsx` at
`./experiments/03-conditional.jsx` and run it. Find the line that reads
`Broken guard (count=0): 0` and the line under it. Same data, different guard. Then look at
the six falsy values at the bottom and check them against the table in this lesson.

**Part 2 — choose the shape.** For each requirement, say early return, ternary or `&&`:

1. A `Price` component that must put nothing on the page at all when the product is
   unavailable.
2. A "Featured" badge that appears only for some products, inside a card that always
   renders.
3. A line that reads either `Online` or `Offline`.
4. A panel that shows a summary normally, but an entirely different "access denied" screen
   for users without permission.

**Part 3 — two questions.**

- `{user.unreadCount && <Dot />}` is wrong. Say what a user sees when there are no unread
  messages, and give **two** different fixes.
- Why can `{cond ? <Dot /> : null}` never leak a stray value, whatever `cond` is?

In [ ]:
// Your code here

## LESSON 19 — Rendering a list with `.map()`

Two facts you already have, and this lesson is what happens when you put them together.

- **LESSON 10:** an array renders each of its items, in order.
- **LESSON 8:** `.map()` is an expression, so it fits inside braces.

**The React API:**

```jsx
const products = ["Bread", "Milk", "Eggs"];

<ul>
  {products.map((name) => <li>{name}</li>)}
</ul>
```

Three strings in, three `<li>` elements out.

### Two separate steps, and only one of them is React

This is the part worth being precise about.

**Step 1 — `.map()` runs.** This is the same `Array.prototype.map` from the JavaScript
course. Nothing about it is React. It takes an array and returns a **new array** of the same
length, where each item is whatever your callback returned. Here the callback returns JSX,
so you get an array of elements — and an element is just an object (LESSON 7).

```js
["Bread", "Milk", "Eggs"]          // data
  ↓ .map(name => <li>{name}</li>)
[<li>Bread</li>, <li>Milk</li>, <li>Eggs</li>]   // an array of elements
```

**Step 2 — React renders it.** React receives that array in the braces and does what
LESSON 10 already told you it does with an array: renders each item, in order.

React's own description of the whole move: *"transform your array of data into an array of
components"*.

### The callback's return value **is** the list item

Whatever the callback gives back becomes the rendered item. Which makes this a very common
bug:

```jsx
{products.map((name) => { <li>{name}</li> })}
```

Those braces are a **function body**, not JSX. There is no `return`, so the callback returns
`undefined` — three times. You get `[undefined, undefined, undefined]`, and `undefined`
renders nothing (LESSON 10). The page is blank where the list should be, with no error.

Two ways to fix it, and both are ordinary JavaScript:

```jsx
{products.map((name) => <li>{name}</li>)}          // implicit return
{products.map((name) => { return <li>{name}</li>; })}  // explicit return
```

### Mapping to a component

The item does not have to be plain markup. Anything you can write in JSX, you can return —
including one of your own components, with props (LESSON 14):

```jsx
{products.map((product) => (
  <ProductRow name={product.name} price={product.price} />
))}
```

Wrapping the returned JSX in parentheses is just formatting: it keeps the arrow's implicit
return while letting the JSX span several lines.

### One thing you will see and cannot fix yet

Render a list this way and React logs a warning:

```text
Each child in a list should have a unique "key" prop.
```

It is right, and it matters. It is also **LESSON 20**, which is entirely about it. For this
lesson, expect the warning and leave it alone.

### Key Notes

- `.map()` is plain JavaScript: data array in, array of elements out. React's part is only
  the rendering, and LESSON 10 already defined that.
- Whatever the **callback returns** becomes the rendered item.
- A block-bodied arrow with no `return` gives `undefined` for every item — a silently empty
  list.
- Expect a `key` warning. That is LESSON 20.

### Example

**Runnable — plain JS.** The `.map()` half is pure JavaScript, so you can watch the array
transform without React anywhere near it. The strings stand in for the elements you would
really be returning.

In [ ]:
const l19products = ["Bread", "Milk", "Eggs"];

// The callback returns something -> that something lands in the new array.
const l19items = l19products.map((name) => `<li>${name}</li>`);
console.log(l19items);
console.log("length in:", l19products.length, "| length out:", l19items.length);

// A block body with no `return` gives back undefined, once per item.
const l19broken = l19products.map((name) => {
  `<li>${name}</li>`;
});
console.log(l19broken);

// React would render each item of the first array, and nothing at all for the second.

### Exercise

Work with this data:

```js
const l19cart = [
  { name: "Bread", price: 2.4 },
  { name: "Milk", price: 1.15 },
  { name: "Eggs", price: 3.8 },
];
```

**Part 1.** Write `l19rows(cart)` which uses `.map()` to return an array of strings, one per
item, each formatted `"Bread — 2.40 EUR"`. Use `.toFixed(2)` for the price. Log the array
and its length.

**Part 2.** Write `l19rowsBroken(cart)` — the same function, but with a block-bodied arrow
and no `return`. Log what it gives back, then answer in a comment: how many `<li>` elements
would React put on the page for this one, and why does nothing appear in the console as an
error?

**Part 3.** In one comment line: `.map()` returns a new array. What does `.forEach()` return,
and why does that make it the wrong method here?

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Part 1 — in the playground.** Point `playground/src/App.jsx` at
`./experiments/04-lists.jsx` and run it. Three lists are rendered. Confirm that the first
two show three rows each and the third shows none, then open the console and read the
warning React logs. Write down what it says — you are not fixing it in this lesson.

**Part 2 — reasoning.** Answer in comments.

1. `{products.map(...)}` puts an **array** inside braces. Which earlier lesson already told
   you what React does with an array, and what did it say?
2. A colleague says "`.map()` is a React thing". Correct them in one sentence.
3. Given `const rows = products.map((p) => <li>{p.name}</li>);` written **above** the
   `return`, what is `rows` at that moment — has anything been rendered yet? (LESSON 7 is
   the one to lean on.)

In [ ]:
// Your code here

## LESSON 20 — Keys: telling React which item is which

LESSON 19 left a warning on your screen:

```text
Each child in a list should have a unique "key" prop.
```

This is what it is for.

### The problem keys solve

When a list re-renders, React has an old array of elements and a new one, and it has to work
out what changed. Did an item move? Was one deleted? Was one inserted at the top?

Without help it can only compare by **position** — first with first, second with second.
React's docs put the problem like this:

> Imagine that files on your desktop didn't have names. Instead, you'd refer to them by
> their order — the first file, the second file, and so on. You could get used to it, but
> once you delete a file, it would get confusing. The second file would become the first
> file, the third file would be the second file, and so on.

A `key` is the file name. It tells React **which item each element belongs to**, so it can
match them up across renders no matter how they move.

```jsx
{products.map((product) => (
  <ProductRow key={product.id} name={product.name} />
))}
```

### Three things that are easy to confuse

| | what it is |
|---|---|
| **the key** | React's identity hint for this item among its siblings. React uses it and then throws it away |
| **the data** | `product.name`, `product.price` — what you actually render |
| **the index** | *where* the item happens to sit in the array right now |

The key is not data. Your component never receives it — React's docs are explicit: *"your
components won't receive `key` as a prop. It's only used as a hint by React itself."* If the
component needs the id as well, pass it a second time as a normal prop:
`<Row key={id} id={id} />`.

### Keys must be stable

A key is only useful if it is the **same value for the same item on every render**. That
gives two rules, both from React's docs:

- **"Keys must be unique among siblings."**
- **"Keys must not change" — don't generate them while rendering.**

So `key={Math.random()}` is the worst possible key: every render invents new names, so React
matches nothing to anything, throws all the DOM away and rebuilds it. The docs note it "will
also lose any user input inside the list items" — which is the same symptom as the next
section, only permanent.

Good keys usually already exist in your data: a database id, a SKU, an order number,
anything the item carries with it and does not lose.

### Why the index breaks — the part you have to see

The index is not a property of the item. It is a property of *the item's position*, and
positions shift the moment anything is removed or inserted.

Here is the experiment, and the numbers are real. Three products, each row holding a text
input the user has typed a note into. The first product is then deleted:

```text
keyed by index                  keyed by id
Milk  => "note-BREAD"           Milk  => "note-MILK"
Eggs  => "note-MILK"            Eggs  => "note-EGGS"
```

With **index keys**, React was told the item at position 0 is still "the item called 0".
Bread's row was never deleted — it was *relabelled* Milk, keeping the DOM node and the text
inside it. Every note slid up one row, and the last note vanished.

With **id keys**, React was told `p1` is gone. It removed that row, kept `p2` and `p3`
exactly as they were, and every note stayed with its own product.

Nothing warned about this. Both versions had keys, so the console was silent. **The bug is
not the warning — the bug is the wrong item identity**, and it only shows when the list
changes.

### When the index is genuinely fine

Do not over-correct. The index is a correct key when all three hold:

- the list never reorders,
- items are never inserted or removed anywhere except the end,
- and the items hold no state of their own — no inputs, no focus, nothing the DOM remembers.

A static list rendered once from fixed data meets all three. Much real data does not.

If your data has no id at all, the honest answer is usually to give it one where it is
created, not to invent one during render.

### Key Notes

- A key tells React **which item** an element is, so it can match items across renders.
- Keys must be **unique among siblings** and **stable** — never generated during render.
- A key is not a prop. The component never receives it.
- The index identifies a **position**, not an item; use it only for lists that never
  reorder, never lose an item from the middle, and hold no state.

### Example

**Runnable — plain JS.** Keys are React's mechanism, but *choosing* one is a data question
you can answer without React. This cell checks a candidate key against the two rules.

In [ ]:
const l20products = [
  { id: "p1", sku: "BRD-01", name: "Bread", price: 2.4 },
  { id: "p2", sku: "MLK-01", name: "Milk", price: 1.15 },
  { id: "p3", sku: "EGG-01", name: "Eggs", price: 3.8 },
  { id: "p4", sku: "BRD-01", name: "Bread (large)", price: 3.9 },
];

function l20isUsableKey(items, pick) {
  const keys = items.map(pick);
  const unique = new Set(keys).size === keys.length;
  const defined = keys.every((k) => k !== undefined && k !== null);
  return { keys, unique, defined, usable: unique && defined };
}

console.log("by id   ->", l20isUsableKey(l20products, (p) => p.id));
console.log("by sku  ->", l20isUsableKey(l20products, (p) => p.sku));
console.log("by name ->", l20isUsableKey(l20products, (p) => p.name));

### Exercise

**Part 1.** Using `l20isUsableKey` from the example, log the verdict for `id`, `sku`, `name`
and `price`. Exactly one of them fails the check — say which and why in a comment.

Then the harder question, also in a comment: **three of them pass, and only one of those
three is a key you would actually use.** Which, and what is the check unable to see?

**Part 2.** Here is the same list after the first product is removed:

```js
const l20after = l20products.slice(1);
```

Write `l20trace(before, after, pick)` which returns, for each item in `after`, a string
showing what a key built with `pick` would have been **before** and **after** the removal:

```text
Milk: key was p2, key is now p2
```

Run it twice — once with `(p) => p.id`, once with `(p, i) => i` — using the index the array
actually had in each list. The id version must show every key unchanged; the index version
must show every key shifted. Log both.

**Part 3.** In a comment, answer: `key={Math.random()}` is unique and never `undefined`, so
`l20isUsableKey` would call it usable. Why is it nonetheless the worst key in this lesson?

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Part 1 — see the identity bug.** Point `playground/src/App.jsx` at
`./experiments/05-keys.jsx` and run it. Both lists render the same three products; only the
key differs.

1. Type a different note into every input, in **both** lists — something you can recognise,
   like `bread-note`.
2. Open `05-keys.jsx` and delete the **first** product from the array. Save.
3. Compare the two lists. In the index-keyed list, which product now holds the note you
   typed for Bread? What happened to the note you typed for Eggs?
4. The console stayed silent through all of that. Say why in one sentence.

**Part 2 — reasoning.** Answer in comments.

1. A colleague fixes the warning with `key={index}` and says it is gone, so the problem is
   solved. What have they actually fixed, and what have they not?
2. You are rendering a list of comments that are only ever appended to the end, never
   reordered or deleted, and each row is plain text with no input. Is `key={index}` wrong
   here? Answer honestly.
3. Your data has no id. Name one place a stable key could come from, and one place it
   must **not** come from.

In [ ]:
// Your code here

## LESSON 21 — Filtering, sorting, counts and empty states

Real lists are rarely rendered raw. You show *some* of the items, in *some* order, usually
with a count beside them. All of that is plain JavaScript, and all of it happens **before**
the JSX — LESSON 8's shape again: work above the `return`, the result in braces.

The important idea is that your data and what you render are two different things. The data
is the truth; what you render is a **view** of it.

```jsx
function ProductList({ products }) {
  const shown = products.filter((p) => p.inStock);

  return (
    <ul>
      {shown.map((p) => <li key={p.id}>{p.name}</li>)}
    </ul>
  );
}
```

`products` is untouched. `shown` is a view.

### Keep the original: one of these methods is not like the others

`.filter()` returns a **new** array. So does `.map()`. `.sort()` does not — it reorders the
array **in place** and hands you back the very same array:

```js
const original = [3, 1, 2];
const result = original.sort();
// original is now [1, 2, 3]   <- it was changed
// result === original         <- true, it is the same array
```

In a component that is a real bug, because the array usually arrived as a prop, and
LESSON 14 was clear: props are read-only. Sorting a prop edits the parent's data.

**Copy first, then sort:**

```js
const sorted = [...products].sort((a, b) => a.price - b.price);
```

The spread makes a new array; the sort rearranges the copy. (Modern JavaScript also has
`toSorted()`, which does both in one call and leaves the original alone. Either is fine —
what matters is that the original survives.)

A comparator is required for numbers, by the way: a bare `.sort()` compares items as
**strings**, so `[10, 9, 1]` sorts to `[1, 10, 9]`.

### Counts are derived, never stored

```js
const shown = products.filter((p) => p.inStock);
const total = products.length;
const hidden = total - shown.length;
```

None of these is a separate piece of information. Each is worked out from the array you
already have, every time the component renders. That matters because a stored count and a
list can disagree — you add an item, forget to bump the number, and the screen says 4 next
to five rows. A value you recompute cannot drift.

The rule to carry forward: **if you can calculate it from what you already have, calculate
it.**

### Empty states

A filter can legitimately match nothing, and an empty array renders **nothing** (LESSON 10).
No error, no message, just a blank space where the list was. That is almost never what you
want the user to see:

**The React API:**

```jsx
function ProductList({ products, query }) {
  const shown = products.filter((p) => p.name.includes(query));

  if (shown.length === 0) {
    return <p>No products match “{query}”.</p>;
  }

  return (
    <ul>
      {shown.map((p) => <li key={p.id}>{p.name}</li>)}
    </ul>
  );
}
```

That is LESSON 18's early return, used for exactly what it is good at: two different trees.

### Filtering data is not conditional rendering

These are easy to blur, and they answer different questions.

| | question it answers |
|---|---|
| **filtering** | *which items* are in the array I am about to render |
| **conditional rendering** | *whether* a piece of UI appears at all |

Both can end with nothing on screen, for different reasons. An empty filtered array renders
nothing because there is nothing left to render. A false condition renders nothing because
you said so.

The empty state is where the two meet: you **filter** the data, then **conditionally render**
a message when the result is empty.

### Key Notes

- The data is the truth; a filtered or sorted array is a **view**, computed above the
  `return`.
- `.filter()` and `.map()` return new arrays. **`.sort()` mutates** — copy first with
  `[...items]`.
- Counts are derived from the array every render, never stored beside it.
- An empty list renders nothing; decide deliberately what the user sees instead.

### Example

**Runnable — plain JS.** The mutation is the part worth seeing rather than believing.

In [ ]:
const l21products = [
  { id: "p1", name: "Bread", price: 2.4, inStock: true },
  { id: "p2", name: "Milk", price: 1.15, inStock: false },
  { id: "p3", name: "Eggs", price: 3.8, inStock: true },
];

// .filter() leaves the original alone.
const l21inStock = l21products.filter((p) => p.inStock);
console.log("filtered:", l21inStock.map((p) => p.name), "| original length:", l21products.length);

// .sort() does not.
const l21names = ["Milk", "Bread", "Eggs"];
const l21sorted = l21names.sort();
console.log("names now:", l21names, "| same array?", l21names === l21sorted);

// Copy first, and the original survives.
const l21prices = [10, 9, 1];
const l21safe = [...l21prices].sort((a, b) => a - b);
console.log("safe copy:", l21safe, "| original untouched:", l21prices);

// A bare .sort() compares as strings.
console.log("no comparator:", [...l21prices].sort());

### Exercise

Work with this data:

```js
const l21catalogue = [
  { id: "a1", name: "Bread", price: 2.4, inStock: true },
  { id: "a2", name: "Milk", price: 1.15, inStock: false },
  { id: "a3", name: "Eggs", price: 3.8, inStock: true },
  { id: "a4", name: "Butter", price: 2.95, inStock: true },
];
```

**Part 1.** Write `l21view(catalogue)` which returns an object describing what a component
would render:

```js
{ shown: [...names, cheapest first...], total: 4, hidden: 1 }
```

- `shown` contains only the in-stock items, sorted by price, cheapest first, as **names**.
- `total` is how many items there are altogether.
- `hidden` is how many were filtered out.

Log the result, then log `l21catalogue` afterwards and confirm the original order is
unchanged.

**Part 2.** Write `l21emptyMessage(shown, query)` which returns the string a user should see:
the empty-state message `No products match "<query>".` when `shown` is empty, and `null`
when there is something to render. Test it both ways.

**Part 3.** In a comment: `total` and `hidden` are numbers you calculated. What would go
wrong if you instead kept a `hiddenCount` variable beside the catalogue and updated it by
hand whenever the catalogue changed?

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

Three questions. Answers in comments; code only where it helps.

1. **Spot the bug.** A component receives `products` as a prop and renders two lists — one
   sorted by price, one in the original order:

   ```jsx
   const byPrice = products.sort((a, b) => a.price - b.price);
   ```

   The second list comes out sorted too, and nobody asked it to. Explain what happened, in
   terms of LESSON 14's rule about props, and give the one-line fix.

2. `{shown.length && <p>{shown.length} results</p>}` — this looks reasonable and has a bug
   you have already met. What does the user see when nothing matches, and why?

3. A colleague suggests storing `hiddenCount` on the product data "so we don't recalculate
   it on every render". Give the strongest argument against it that does not mention
   performance.

In [ ]:
// Your code here